# 📊 IPL Match Data Extraction Pipeline

This notebook implements a structured ETL pipeline for the **IPL Match Ball-by-Ball Dataset** sourced from **Cricsheet JSON files**.

The dataset contains **1000+ IPL match JSON files**, each storing detailed match data including:

- Match metadata  
- Teams  
- Venue  
- Toss results  
- Player of the match  
- Ball-by-ball deliveries  
- Runs and wickets  

Each JSON file represents **one complete IPL match** with nested innings and delivery-level events.

## 🎯 Objectives

This pipeline performs:

1. Recursive JSON file traversal  
2. Match metadata extraction  
3. Ball-by-ball delivery extraction  
4. Match-level aggregation  
5. Over-level aggregation  
6. Data cleaning & standardization  
7. CSV export for Power BI modeling  

The final structured datasets are prepared for **Star Schema modeling in Power BI**.

In [ ]:
import os
import json
import pandas as pd
from pathlib import Path

## ⚙️ Configuration

Define the base directory containing the IPL JSON files downloaded from **Cricsheet**.

Ensure the dataset folder exists in the same directory as this notebook.

Example folder structure: ipl_male_json/

In [ ]:
BASE_DIR = Path("ipl_male_json")

if not BASE_DIR.exists():
    print("❌ IPL JSON directory not found.")
else:
    print("✅ IPL JSON directory located successfully.")

## 🔹 Phase 1: Match Metadata Extraction

This section extracts **match-level information** including:

- Match ID  
- Season  
- Match Date  
- Team 1  
- Team 2  
- Venue  
- City  
- Toss Winner  
- Toss Decision  
- Match Winner  
- Player of the Match  

These values are stored inside the **match metadata block** in each JSON file.

The extracted data forms the **match table used in the Power BI data model**.

In [ ]:
def extract_match_metadata(base_dir):

    matches = []

    for file in os.listdir(base_dir):

        if not file.endswith(".json"):
            continue

        with open(base_dir / file, "r") as f:
            data = json.load(f)

        info = data["info"]
        match_id = file.replace(".json","")

        try:
            matches.append({
                "match_id": match_id,
                "season": info.get("season"),
                "date": info.get("dates")[0],
                "team1": info["teams"][0],
                "team2": info["teams"][1],
                "venue": info.get("venue"),
                "city": info.get("city"),
                "toss_winner": info["toss"]["winner"],
                "toss_decision": info["toss"]["decision"],
                "winner": info.get("outcome", {}).get("winner"),
                "player_of_match": info.get("player_of_match",[None])[0]
            })

        except KeyError:
            continue

    return pd.DataFrame(matches)

In [ ]:
print("🔄 Extracting IPL Match Metadata...")

df_matches = extract_match_metadata(BASE_DIR)

if not df_matches.empty:
    df_matches.to_csv("ipl_matches.csv", index=False)
    print("✅ Saved as ipl_matches.csv")
    print("Rows Extracted:", len(df_matches))
else:
    print("⚠️ Extraction returned empty dataframe.")

## 🔹 Phase 2: Ball-by-Ball Delivery Extraction

This phase extracts **ball-level match events**.

Each JSON file contains nested data structured as: innings → overs → deliveries

Fields extracted:

- Match ID  
- Batting Team  
- Over Number  
- Ball Number  
- Batter  
- Bowler  
- Runs scored by batter  
- Extras  
- Total runs  
- Wicket indicator  

This dataset forms the **core fact table for cricket analytics**.


In [ ]:
def extract_deliveries(base_dir):

    deliveries = []

    for file in os.listdir(base_dir):

        if not file.endswith(".json"):
            continue

        with open(base_dir / file, "r") as f:
            data = json.load(f)

        match_id = file.replace(".json","")

        for inning in data["innings"]:

            for team, inning_data in inning.items():

                batting_team = inning_data["team"]

                for over in inning_data["overs"]:

                    over_number = over["over"]

                    for ball in over["deliveries"]:

                        deliveries.append({
                            "match_id": match_id,
                            "batting_team": batting_team,
                            "over": over_number,
                            "ball": ball["ball"],
                            "batter": ball["batter"],
                            "bowler": ball["bowler"],
                            "runs_batter": ball["runs"]["batter"],
                            "extras": ball["runs"]["extras"],
                            "total_runs": ball["runs"]["total"],
                            "wicket_flag": 1 if "wickets" in ball else 0
                        })

    return pd.DataFrame(deliveries)

In [ ]:
print("🔄 Extracting Ball-by-Ball Deliveries...")

df_deliveries = extract_deliveries(BASE_DIR)

if not df_deliveries.empty:
    df_deliveries.to_csv("ipl_deliveries.csv", index=False)
    print("✅ Saved as ipl_deliveries.csv")
    print("Rows Extracted:", len(df_deliveries))
else:
    print("⚠️ No delivery data extracted.")

## 🔹 Phase 3: Match Summary Aggregation

This phase aggregates deliveries to create **match-level statistics**.

Metrics generated:

- Total runs per innings  
- Total wickets per innings  

These statistics enable:

- Scorecard reconstruction  
- Win margin calculations  
- Strategic match analysis

In [ ]:
def create_match_summary(df_deliveries):

    match_summary = (
        df_deliveries
        .groupby(["match_id","batting_team"])
        .agg(
            total_runs=("total_runs","sum"),
            wickets=("wicket_flag","sum")
        )
        .reset_index()
    )

    return match_summary

In [ ]:
print("🔄 Creating Match Summary Table...")

df_match_summary = create_match_summary(df_deliveries)

df_match_summary.to_csv("ipl_match_summary.csv", index=False)

print("✅ Saved as ipl_match_summary.csv")

## 🚀 ETL Pipeline Completed Successfully

Generated datasets:

- `ipl_matches.csv`
- `ipl_deliveries.csv`
- `ipl_match_summary.csv`
- `ipl_over_summary.csv`

These datasets are now ready for:

- Star Schema Data Modeling  
- Power BI Analytics  
- Player Intelligence  
- Franchise Intelligence  
- Match Strategy Analysis